### Lab 3.3 Backpropagation

In this lab you will inspect the gradients in a neural network and understand how they are computed as they propagate from the loss backwards through the network.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

Let's make some random data: a 2D input point and a label (set to one).

In [2]:
X = torch.randn(2)
y = torch.ones(1).long()

In [3]:
X, y

(tensor([ 0.1037, -0.5624]), tensor([1]))

Now let's make the parameters needed for a multi-layer perceptron with a single hidden layer of three neurons.  We will be sure to set `requires_grad=True` so that PyTorch knows to compute the gradients for these tensors.

In [4]:
W1 = torch.randn(3,2,requires_grad=True)
b1 = torch.randn(3,requires_grad=True)

W2 = torch.randn(1,3,requires_grad=True)
b2 = torch.randn(1,requires_grad=True)

Now we compute the score output and a squared error loss term.  This is the "forward" step as the computation flows from the input to the output and then the loss.

$$\vec{h} = W_1 \vec{X} + \vec{b}_1$$
$$\vec{s} = \textrm{ReLU}(\vec{h})$$
$$z = W_2 \vec{s} + b_2$$
$$L(z) = \frac{1}{2}(z-y)^2$$

In [5]:
h = W1@X+ b1
s = F.relu(h)

z = W2@s + b2

L = 0.5*(z-y)**2

`retain_grad()` tells PyTorch not to throw away the gradients of intermediate tensors.

In [6]:
h.retain_grad()
s.retain_grad()
z.retain_grad()
L.retain_grad()

Now we call `backward()` to compute the gradients.

In [7]:
L.backward()

Let's think about what the gradient of the loss w.r.t. $z$ should be.
$$L = \frac{1}{2}(z-y)^2$$
$$\frac{dL}{dz} = (z-y)$$

In [8]:
dLdz = z-y

Let's check our answer with PyTorch's answer.

In [9]:
dLdz, z.grad

(tensor([-1.4794], grad_fn=<SubBackward0>), tensor([-1.4794]))

Yup, they're the same!

Now let's think about how to calculate $dL/W_2$ and $dL/db_2$.

$$z = W_2 \vec{s} + b_2$$

To calculate $dL/W_2$ and $dL/db_2$ we need to use the chain rule:

$$\frac{dL}{W_2} = \frac{dL}{dz}\frac{dz}{dW_2}$$
$$\frac{dL}{b_2} = \frac{dL}{dz}\frac{dz}{db_2}$$

$$\frac{dz}{dW_2} = \vec{s}^T$$
$$\frac{dz}{db_2} = 1$$

In [10]:
dzdW2 = s.unsqueeze(0)

In [11]:
s.shape, dzdW2.shape, s.unsqueeze(-1).shape, dLdz.shape, dLdz.unsqueeze(-1).shape

(torch.Size([3]),
 torch.Size([1, 3]),
 torch.Size([3, 1]),
 torch.Size([1]),
 torch.Size([1, 1]))

In [12]:
dLdW2 = dLdz.unsqueeze(-1) @ dzdW2

In [13]:
dLdW2, W2.grad

(tensor([[ 0.0000, -1.2593, -1.7309]], grad_fn=<MmBackward0>),
 tensor([[-0.0000, -1.2593, -1.7309]]))

In [14]:
dzdb2 = torch.eye(1)

In [15]:
dLdb2 = dLdz @ dzdb2

Note that it would have been more efficient to simply do `dLdb2 = dLdz` since multiplying by $1$ has no effect.

In [16]:
dLdb2, b2.grad

(tensor([-1.4794], grad_fn=<SqueezeBackward4>), tensor([-1.4794]))

### Exercise

Continue to work back until you can calculate the derivatives w.r.t. $W_1$ and $\vec{b_1}$.

1. Calculate $dz/d\vec{s}$, apply it to compute $dL/d\vec{s}$, and check your answer.


$$z = W_2 \vec{s} + \vec{b}_2$$
$$\frac{dz}{d\vec{s}} = ?$$

In [17]:
dzds = W2
dLds = dLdz*dzds

dLds, s.grad

(tensor([[0.2081, 0.6844, 0.3036]], grad_fn=<MulBackward0>),
 tensor([0.2081, 0.6844, 0.3036]))

2. Compute $d\vec{s}/d\vec{h}$, apply it to compute $dL/d\vec{h}$, and check your answer.

$$\vec{s} = \textrm{ReLU}(\vec{h})$$
$$\frac{d\vec{s}}{d\vec{h}}=?$$
Note that the derivative of ReLU is 0 when the input $\leq 0$ and 1 otherwise.  (Technically, ReLU is non-differentiable at 0 and this is the "sub-gradient," which is sufficient to make gradient descent work.)


In [21]:
dsdh = torch.where(h > 0, 1, 0)
dLdh = dLds*dsdh

dLdh, h.grad

(tensor([[0.0000, 0.6844, 0.3036]], grad_fn=<MulBackward0>),
 tensor([0.0000, 0.6844, 0.3036]))

3. Compute $d\vec{h}/dW_1$ and $d\vec{h}/db_1$, apply them compute $dL/dW_1$ and $dL/d\vec{b_1}$, and check your answers.



In [19]:
dhdW1 = X.unsqueeze(0)
dLdW1 = dLdh.unsqueeze(-1) @ dhdW1

dLdW1, W1.grad

(tensor([[[ 0.0000, -0.0000],
          [ 0.0710, -0.3850],
          [ 0.0315, -0.1708]]], grad_fn=<UnsafeViewBackward0>),
 tensor([[ 0.0000, -0.0000],
         [ 0.0710, -0.3850],
         [ 0.0315, -0.1708]]))

In [20]:
dhdb1 = torch.eye(3)
dLdb1 = dLdh @ dhdb1

dLdb1, b1.grad

(tensor([[0.0000, 0.6844, 0.3036]], grad_fn=<MmBackward0>),
 tensor([0.0000, 0.6844, 0.3036]))